In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("logs/evaluate/data_frame.csv")

In [ ]:
AGGREGATION = ("mean", "std")
df_grouped = df.groupby(
    [
        "boundary",
        "strategy",
        "detector",
        "detector_label",
    ]
).aggregate(AGGREGATION)

# Turn the multi-index columns into columns of tuples
df_grouped

In [ ]:
from typing import Optional, Literal
from jinja2 import Template

_template = Template(
    r"{% if font_style %}\{{font_style ~ '{'}}{% endif %}"
    r"{{ '{:.{}f}'.format(mean, ndigits) }}\tiny{$\pm${{ '{:.{}f}'.format(std, ndigits) ~ '}' }}"
    r"{% if font_style %}}{% endif %}"
)

TexFontStyle = Literal["textmd", "textbf", "textup", "textit", "textsl", "textsc"]


def format_column(
    df: pd.DataFrame | pd.Series,
    mean_col="mean",
    std_col="std",
    min_style: Optional[TexFontStyle] = None,
    max_style: Optional[TexFontStyle] = None,
    ndigits=3,
    nan_str="N/A",
) -> pd.DataFrame | pd.Series:
    col_max = df[mean_col].max()
    col_min = df[mean_col].min()

    def _func(row: pd.Series) -> str:
        mean = float(row[mean_col])
        std = float(row[std_col])
        font_style = None

        if pd.isna(mean) or pd.isna(std):
            return nan_str

        # Perform the comparison using the same precision as the formatted output
        if max_style and round(mean, ndigits) == round(col_max, ndigits):
            font_style = max_style
        elif min_style and round(mean, ndigits) == round(col_min, ndigits):
            font_style = min_style

        return _template.render(
            mean=mean,
            std=std,
            font_style=font_style,
            ndigits=ndigits,
        )

    return df.apply(_func, axis=1)

In [ ]:
from pathlib import Path

BOUNDARY_AGNOSTIC = {"ER", "FT"}


def latex_table(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # index to columns
    df.reset_index(inplace=True)
    # Move the ER and FT rows to the end of the table, since they are boundary-agnostic
    boundary_agnostic_mask = df["strategy"].isin(BOUNDARY_AGNOSTIC)
    df = pd.concat(
        [df[~boundary_agnostic_mask], df[boundary_agnostic_mask]], ignore_index=True
    )

    # Ignore drift detection metrics (dd.*) for the oracle detector, since they are not
    # meaningful
    oracle_mask = df["detector"] == "oracle"
    dd_cols = [col for col in df.columns if col[0].startswith("dd.")]
    df.loc[oracle_mask, dd_cols] = np.nan

    ndigits_percent = 1
    UP = r"$\uparrow$"
    DOWN = r"$\downarrow$"

    # Create a copy of the data frame with just the index
    df_formatted = pd.DataFrame(index=df.index)
    df_formatted["Strategy"] = df["strategy"]
    df_formatted["Detector"] = df["detector"]

    df_formatted[f"Acc. {UP}"] = format_column(
        df["ocl.accuracy_seen_avg"] * 100, max_style="textbf", ndigits=ndigits_percent
    )
    df_formatted[f"FWT {UP}"] = format_column(
        df["ocl.forward_transfer"] * 100, max_style="textbf", ndigits=ndigits_percent
    )
    df_formatted[f"BWT {UP}"] = format_column(
        df["ocl.backward_transfer"] * 100, max_style="textbf", ndigits=ndigits_percent
    )
    df_formatted[f"F1 {UP}"] = format_column(
        df["dd.f1"] * 100, max_style="textbf", ndigits=ndigits_percent
    )
    df_formatted[f"WD {DOWN}"] = format_column(
        df["dd.wasserstein_distance"] * 100, min_style="textbf", ndigits=ndigits_percent
    )
    df_formatted[f"MDT {DOWN}"] = format_column(
        df["dd.mdt"], max_style="textbf", ndigits=0
    )

    return df_formatted


def table_to_latex(df: pd.DataFrame, filename: Path):
    df_formatted = latex_table(df)
    latex = df_formatted.to_latex(index=False).splitlines()
    # Add a midrule before the boundary-agnostic strategies
    latex.insert(len(latex) - 2 - len(BOUNDARY_AGNOSTIC), r"\midrule")
    latex_str = "\n".join(latex)
    filename.write_text(latex_str)
    return df_formatted


table_to_latex(df_grouped.loc["abrupt"], Path("table/strategy_abrupt.tex"))  # type: ignore
table_to_latex(df_grouped.loc["gradual"], Path("table/strategy_gradual.tex"))  # type: ignore
table_to_latex(df_grouped.loc["slow"], Path("table/strategy_slow.tex"))  # type: ignore